# Avaliação do sistema hierárquico no conjunto de teste adicional

## Bibliotecas utilizadas

In [2]:
import numpy as np
import os
import re
import shutil
from matplotlib import pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, fbeta_score

## Definição das classes

In [3]:
actions = ['Alergia', 'Autismo', 'Bronquite', 'Caimbra', 'Convulsao', 'Diarreia', 'Fratura', 'Frieira', 'Infarto', 'Sarampo']
label_map = {label: num for num, label in enumerate(actions)}
num_frames = 30

#Classes que o modelo de 7 classes pode prever (as classes unidas, em ordem alfabética)
classes_geral = ['Alergia_Frieira', 'Autismo', 'Bronquite_Infarto', 'Caimbra_Fratura', 'Convulsao', 'Diarreia', 'Sarampo']

#Especialista de cada classe unida: as duas classes que ele separa e a pasta dos pesos
especialistas = {
    'Alergia_Frieira': {'classes': ['Alergia', 'Frieira'], 'pasta': 'ale_fri'},
    'Caimbra_Fratura': {'classes': ['Caimbra', 'Fratura'], 'pasta': 'cai_fra'},
    'Bronquite_Infarto': {'classes': ['Bronquite', 'Infarto'], 'pasta': 'bro_inf'}
}

## Parâmetros e pastas

In [ ]:
weights_geral_dir = "../models/models/7_classes"
weights_especialistas_dir = "../models/models"
results_dir = "../results/hierarchical_test"
cm_dir = os.path.join(results_dir, "matrizes_por_conjunto")
os.makedirs(cm_dir, exist_ok=True)

#Keypoints de todos os frames de cada vídeo do conjunto de teste adicional e pasta dos conjuntos sorteados
test_dir = "../keypoints_data_test"
test_sorted_dir = "../keypoints_data_test_sorted"

#Pasta do conjunto do especialista de Alergia e Frieira (início do movimento detectado em cada vídeo)
ale_fri_test_dir = "../keypoints_data_ale_fri_test"

num_endings = 20 #Quantidade de conjuntos de teste sorteados de cada vídeo

usar_latex = True #Colocar False se não tiver o Latex instalado

## Sorteio dos conjuntos de teste

In [5]:
def natural_sort_key(fname):
    #Ordenação correta para números de texto, então 10.npy vem depois de 2.npy
    parts = re.split(r'(\d+)', fname)
    return [int(p) if p.isdigit() else p.lower() for p in parts]


def sample_frames_normal(video_dir, target_count):
    #Sorteia target_count frames do vídeo, com mais chance para os frames do meio (distribuição normal)
    files = sorted(
        (f for f in os.listdir(video_dir) if f.endswith('.npy')),
        key=natural_sort_key
    )
    N = len(files)
    if N < target_count:
        raise ValueError(f"Vídeo {video_dir} tem apenas {N} frames, necessário ≥ {target_count}")

    mu = (N - 1) / 2.0 #Centro da distribuição: frame do meio
    sigma = mu * 0.4 #Desvio padrão: 40% da distância até o centro

    idxs = np.arange(N)
    pdf = np.exp(-0.5 * ((idxs - mu) / sigma)**2)
    probs = pdf / pdf.sum()

    chosen = np.random.choice(idxs, size=target_count, replace=False, p=probs)
    chosen.sort()
    return chosen, files


#Cada execução apaga os conjuntos anteriores e sorteia 20 novos de cada vídeo (pasta test_sorted_dir/classe/video_k, k = conjunto)
shutil.rmtree(test_sorted_dir, ignore_errors=True)

pad = len(str(num_frames - 1)) #Largura do zero à esquerda no nome dos arquivos (00.npy, 01.npy, ...)

for action in actions:
    action_dir = os.path.join(test_dir, action)
    for video_name in sorted(os.listdir(action_dir)):
        video_path = os.path.join(action_dir, video_name)

        for k in range(1, num_endings + 1):
            chosen_idxs, files = sample_frames_normal(video_path, num_frames)

            set_dir = os.path.join(test_sorted_dir, action, f"{video_name}_{k}")
            os.makedirs(set_dir, exist_ok=True)

            for new_i, old_i in enumerate(chosen_idxs):
                shutil.copy(os.path.join(video_path, files[old_i]), os.path.join(set_dir, f"{new_i:0{pad}d}.npy"))

## Carregamento dos dados de teste

In [6]:
#Dados de teste de cada conjunto
#X: janelas de keypoints dos vídeos
#y: rótulos dos vídeos
#names: nome da pasta de cada vídeo, para achar o mesmo vídeo no conjunto do especialista de Alergia e Frieira
test_sets = {k: {'X': [], 'y': [], 'names': []} for k in range(1, num_endings+1)}

for action in actions:
    action_dir = os.path.join(test_sorted_dir, action)
    for video_folder in os.listdir(action_dir):
        #Identifica o conjunto pelo número no fim do nome (ex.: Alergia_1_3 -> conjunto 3)
        k = int(re.search(r'_(\d+)$', video_folder).group(1))

        video_dir = os.path.join(action_dir, video_folder)
        window = []
        for frame_num in range(num_frames):
            fname = f"{frame_num:0{pad}d}.npy"
            fpath = os.path.join(video_dir, fname)
            if not os.path.exists(fpath):
                break
            res = np.load(fpath)
            window.append(res)

        if len(window) == num_frames:
            test_sets[k]['X'].append(window)
            test_sets[k]['y'].append(label_map[action])
            test_sets[k]['names'].append(video_folder)

#Converte para array e transforma os rótulos em one-hot
for k in range(1, num_endings+1):
    test_sets[k]['X'] = np.array(test_sets[k]['X'])
    test_sets[k]['y'] = to_categorical(test_sets[k]['y'], num_classes=len(actions))

## Detecção do início do movimento (especialista de Alergia e Frieira)

In [7]:
def adaptive_motion_start(motion_energy, min_baseline_frames=5, second_window=(10, 20), N=5):
    #Início do movimento: primeiro trecho de N frames com energia acima do limiar (média + desvio dos primeiros frames)
    motion_idx = 0
    baseline = np.mean(motion_energy[:min_baseline_frames]) + np.std(motion_energy[:min_baseline_frames])

    for i in range(len(motion_energy) - N):
        if np.all(motion_energy[i:i+N] > baseline):
            motion_idx = i
            break

    #Se o movimento já começa nos primeiros frames, recalcula o limiar com uma janela mais adiante
    if motion_idx <= min_baseline_frames:
        start, end = second_window
        if end < len(motion_energy):
            baseline = np.mean(motion_energy[start:end]) + 2 * np.std(motion_energy[start:end])
            for i in range(len(motion_energy) - N):
                if np.all(motion_energy[i:i+N] > baseline):
                    motion_idx = i
                    break

    return motion_idx, baseline


def detect_motion_start(video_dir, target_count, step):
    #Retorna target_count frames a partir do início do movimento, pegando um a cada step frames
    files = sorted([f for f in os.listdir(video_dir) if f.endswith(".npy")], key=natural_sort_key)
    if len(files) == 0:
        print(f"[{video_dir}] Nenhum arquivo .npy encontrado.")
        return [], 0

    keypoints = []
    for f in files:
        arr = np.load(os.path.join(video_dir, f))
        if arr.ndim == 1:
            arr = arr.reshape(-1, 3)
        keypoints.append(arr)
    keypoints = np.stack(keypoints)

    #Índices das mãos esquerda e direita (os 33 primeiros pontos são da pose)
    lh_indices = list(range(33, 54))
    rh_indices = list(range(54, 75))

    #Centro de cada mão em cada frame (NaN quando alguma das mãos não foi detectada)
    lh_centers, rh_centers = [], []
    for frame in keypoints:
        lh = frame[lh_indices, :]
        rh = frame[rh_indices, :]
        if np.all(lh == 0) or np.all(rh == 0):
            lh_centers.append([np.nan, np.nan])
            rh_centers.append([np.nan, np.nan])
            continue
        lh_centers.append(np.nanmean(lh[:, :2], axis=0))
        rh_centers.append(np.nanmean(rh[:, :2], axis=0))

    lh_centers = np.array(lh_centers)
    rh_centers = np.array(rh_centers)

    #Mantém só os frames em que as duas mãos foram detectadas
    valid_mask = ~np.isnan(lh_centers[:, 0]) & ~np.isnan(rh_centers[:, 0])
    if not np.any(valid_mask):
        print(f"[{video_dir}] nenhum frame válido após filtrar zeros.")
        return files[:target_count], 0

    valid_files = [f for i, f in enumerate(files) if valid_mask[i]]
    lh_centers = lh_centers[valid_mask]
    rh_centers = rh_centers[valid_mask]

    if len(lh_centers) < 5:
        print(f"[{video_dir}] poucos frames válidos ({len(lh_centers)}).")
        return valid_files[:target_count], 0

    #Posição vertical das mãos suavizada (Savitzky-Golay) e energia do movimento (soma das velocidades)
    lh_y_smooth = savgol_filter(lh_centers[:, 1], window_length=5, polyorder=3)
    rh_y_smooth = savgol_filter(rh_centers[:, 1], window_length=5, polyorder=3)
    motion_energy = np.abs(np.gradient(lh_y_smooth)) + np.abs(np.gradient(rh_y_smooth))

    motion_idx, baseline = adaptive_motion_start(motion_energy)

    #Frames a partir do início do movimento, um a cada step (repete o último se faltar)
    chosen_files = valid_files[motion_idx:motion_idx + target_count * step : step]
    if len(chosen_files) < target_count and len(chosen_files) > 0:
        chosen_files += [chosen_files[-1]] * (target_count - len(chosen_files))
    elif len(chosen_files) == 0:
        chosen_files = valid_files[:target_count]

    print(f"[{os.path.basename(video_dir)}] início movimento (adapt): {motion_idx}")
    return chosen_files, motion_idx


step = 2 #Intervalo entre os frames escolhidos (2 = um frame a cada dois)

#Cada vídeo de Alergia e Frieira gera uma pasta com 30 frames a partir do início do movimento (pasta ale_fri_test_dir/classe/video)
shutil.rmtree(ale_fri_test_dir, ignore_errors=True)

for action in ['Alergia', 'Frieira']:
    action_dir = os.path.join(test_dir, action)
    for video_name in sorted(os.listdir(action_dir)):
        chosen_files, motion_idx = detect_motion_start(os.path.join(action_dir, video_name), num_frames, step)

        out_dir = os.path.join(ale_fri_test_dir, action, video_name)
        os.makedirs(out_dir, exist_ok=True)

        for new_i, fname in enumerate(chosen_files):
            shutil.copy(os.path.join(action_dir, video_name, fname), os.path.join(out_dir, f"{new_i:0{pad}d}.npy"))

[Alergia_1] início movimento (adapt): 42
[Alergia_2] início movimento (adapt): 54
[Alergia_3] início movimento (adapt): 9
[Alergia_4] início movimento (adapt): 49
[Alergia_5] início movimento (adapt): 55
[Alergia_6] início movimento (adapt): 16
[Frieira_1] início movimento (adapt): 37
[Frieira_2] início movimento (adapt): 6
[Frieira_3] início movimento (adapt): 19
[Frieira_4] início movimento (adapt): 68
[Frieira_5] início movimento (adapt): 26
[Frieira_6] início movimento (adapt): 29


## Dados do especialista de Alergia e Frieira

In [8]:
#Cada vídeo é identificado pelo nome da pasta, sem o número do conjunto (ex.: Alergia_1)
ale_fri_test = {}

for action in ['Alergia', 'Frieira']:
    action_dir = os.path.join(ale_fri_test_dir, action)
    for video_folder in os.listdir(action_dir):
        video_dir = os.path.join(action_dir, video_folder)
        window = []
        for frame_num in range(num_frames):
            fname = f"{frame_num:0{pad}d}.npy"
            fpath = os.path.join(video_dir, fname)
            if not os.path.exists(fpath):
                break
            res = np.load(fpath)
            window.append(res)

        if len(window) == num_frames:
            ale_fri_test[video_folder] = np.array(window)

## Carregamento dos modelos

In [9]:
#Modelo final de 7 classes e modelo final de cada especialista (treinados com todos os dados)
modelo_geral = load_model(f'{weights_geral_dir}/model_final_all.keras')

modelos_especialistas = {
    classe_unida: load_model(f"{weights_especialistas_dir}/{info['pasta']}/model_final_{info['pasta']}.keras")
    for classe_unida, info in especialistas.items()
}

E0000 00:00:1791809161.844220   27564 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1791809161.853367   27564 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## Funções auxiliares

In [10]:
def class_acc(true, pred, labels):
    #Retorna a acurácia de cada classe
    acc = {}
    for i, lbl in enumerate(labels):
        idx = np.where(true == i)[0]
        acc[lbl] = np.mean(pred[idx] == true[idx]) if len(idx) > 0 else 0
    return acc


labels = list(label_map.keys())


def salva_matriz(cm, caminho):
    #Salva em PDF a matriz de confusão normalizada por linha, com a porcentagem e a quantidade de amostras em cada célula
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    #Porcentagens com duas casas decimais e vírgula (ex.: 76,00)
    percent_str = np.vectorize(lambda x: f"{x:0.2f}".replace(".", ","))(np.round(cm_norm * 100, 2))

    #Símbolo de porcentagem (com LaTeX é preciso usar a barra)
    simbolo = "\\%" if usar_latex else "%"

    #Texto de cada célula: porcentagem e quantidade de amostras (ex.: 76,00% e 38)
    annot_text = np.empty_like(percent_str, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot_text[i, j] = f"{percent_str[i, j]}{simbolo}\n{cm[i, j]}"

    labels_plot = actions.copy()
    labels_plot[3] = 'Câimbra'
    labels_plot[4] = 'Convulsão'

    #Fontes LaTeX nas figuras (requer LaTeX instalado)
    with plt.rc_context({"text.usetex": usar_latex, "font.family": "serif"}):
        plt.figure(figsize=(10,9))

        ax = sns.heatmap(
            cm_norm,
            annot=annot_text,
            fmt='',
            cmap='Blues',
            xticklabels=labels_plot,
            yticklabels=labels_plot,
            cbar=False,
            annot_kws={"size": 14},
            square=True
        )

        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_title("")
        plt.xticks(rotation=90)
        plt.yticks(rotation=0)
        ax.set_xticklabels(ax.get_xticklabels(), fontsize=14)
        ax.set_yticklabels(ax.get_yticklabels(), fontsize=14)

        plt.tight_layout()
        plt.savefig(caminho)
        plt.close()


In [11]:
def preve_sistema(X_test, y_true, nomes):
    #Retorna a classe final (índice nas 10 classes) que o sistema prevê para cada vídeo
    #1) O modelo de 7 classes prevê uma classe, que pode ser uma classe unida (ex.: Alergia_Frieira)
    #2) Se for uma classe unida e o vídeo for de uma das duas classes, o especialista decide entre elas
    #3) Se o vídeo não for de nenhuma das duas, o sistema já errou e a previsão fica sendo a primeira classe do par
    pred_geral = np.argmax(modelo_geral.predict(X_test), axis=1)

    y_pred = np.zeros(len(X_test), dtype=int)
    pendentes = {classe_unida: [] for classe_unida in especialistas} #Vídeos que vão para cada especialista

    for idx, p in enumerate(pred_geral):
        classe_geral = classes_geral[p]

        if classe_geral not in especialistas:
            y_pred[idx] = label_map[classe_geral]
            continue

        classes_par = especialistas[classe_geral]['classes']
        if actions[y_true[idx]] in classes_par:
            pendentes[classe_geral].append(idx)
        else:
            y_pred[idx] = label_map[classes_par[0]]

    for classe_unida, idxs in pendentes.items():
        if len(idxs) == 0:
            continue

        if classe_unida == 'Alergia_Frieira':
            #Mesmo vídeo, mas com o início do movimento detectado (o nome da pasta, sem o número do conjunto, identifica o vídeo)
            X_esp = np.array([ale_fri_test[re.sub(r'_\d+$', '', nomes[idx])] for idx in idxs])
        else:
            X_esp = X_test[idxs]

        pred_esp = np.argmax(modelos_especialistas[classe_unida].predict(X_esp), axis=1)
        for idx, p in zip(idxs, pred_esp):
            y_pred[idx] = label_map[especialistas[classe_unida]['classes'][p]]

    return y_pred

## Avaliação dos 20 conjuntos de teste

In [12]:
set_accs = [] #Acurácia geral de cada conjunto

#Métricas macro (média entre as classes) de cada conjunto
macro_precision_sets = []
macro_recall_sets = []
macro_f1_sets = []
macro_f2_sets = []

#Métricas por classe de cada conjunto
per_class_precision = {lbl: [] for lbl in labels}
per_class_recall = {lbl: [] for lbl in labels}
per_class_f1 = {lbl: [] for lbl in labels}
per_class_f2 = {lbl: [] for lbl in labels}

for k in range(1, num_endings + 1):
    print(f"\n========== Conjunto {k} ==========")

    X_test = test_sets[k]['X']
    y_true = np.argmax(test_sets[k]['y'], axis=1)
    y_pred = preve_sistema(X_test, y_true, test_sets[k]['names'])

    #Acurácia geral do conjunto
    acc_k = np.mean(y_pred == y_true)
    set_accs.append(acc_k)
    print(f"Acurácia geral do conjunto {k}: {acc_k:.4f}")

    #Precisão, recall, F1 e F2 por classe
    label_ids = list(range(len(labels)))
    precisions_k = precision_score(y_true, y_pred, labels=label_ids, average=None, zero_division=0)
    recalls_k = recall_score(y_true, y_pred, labels=label_ids, average=None, zero_division=0)
    f1s_k = f1_score(y_true, y_pred, labels=label_ids, average=None, zero_division=0)
    f2s_k = fbeta_score(y_true, y_pred, beta=2, labels=label_ids, average=None, zero_division=0)

    for i, lbl in enumerate(labels):
        per_class_precision[lbl].append(precisions_k[i])
        per_class_recall[lbl].append(recalls_k[i])
        per_class_f1[lbl].append(f1s_k[i])
        per_class_f2[lbl].append(f2s_k[i])

    #Métricas macro (média entre as classes)
    macro_precision_sets.append(precision_score(y_true, y_pred, average='macro', zero_division=0))
    macro_recall_sets.append(recall_score(y_true, y_pred, average='macro', zero_division=0))
    macro_f1_sets.append(f1_score(y_true, y_pred, average='macro', zero_division=0))
    macro_f2_sets.append(fbeta_score(y_true, y_pred, beta=2, average='macro', zero_division=0))

    #Matriz de confusão do conjunto, salva em results_dir/matrizes_por_conjunto
    cm = confusion_matrix(y_true, y_pred, labels=label_ids)
    salva_matriz(cm, os.path.join(cm_dir, f'conjunto_{k}.pdf'))


========== Conjunto 1 ==========
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 244ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step
Acurácia geral do conjunto 1: 0.9667

========== Conjunto 2 ==========
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 247ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Acurácia geral do conjunto 2: 1.0000

========== Conjunto 3 ==========
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 231ms/step
Acurácia geral do conjunto 3: 0.9500

========== Conjunto 4 ==========
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
Acurácia geral do conjunto 4: 0.9667

========== Conjunto 5 ==========
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 2

### Resultado entre os conjuntos

In [13]:
print("================ RESULTADO FINAL ================")
print(f"Acurácia geral: {np.mean(set_accs) * 100:.2f} ± {np.std(set_accs) * 100:.2f}")

print("\n=========== MÉTRICAS MACRO FINAIS ===========")
print(f"Precisão: média = {np.mean(macro_precision_sets):.4f} | desvio = {np.std(macro_precision_sets):.4f}")
print(f"Recall:   média = {np.mean(macro_recall_sets):.4f} | desvio = {np.std(macro_recall_sets):.4f}")
print(f"F1:       média = {np.mean(macro_f1_sets):.4f} | desvio = {np.std(macro_f1_sets):.4f}")
print(f"F2:       média = {np.mean(macro_f2_sets):.4f} | desvio = {np.std(macro_f2_sets):.4f}")

print("\n=========== MÉTRICAS POR CLASSE (MÉDIA ± STD) ===========")
for lbl in labels:
    print(lbl)
    print(f"  Precisão: {np.mean(per_class_precision[lbl]):.4f} ± {np.std(per_class_precision[lbl]):.4f}")
    print(f"  Recall:   {np.mean(per_class_recall[lbl]):.4f} ± {np.std(per_class_recall[lbl]):.4f}")
    print(f"  F1:       {np.mean(per_class_f1[lbl]):.4f} ± {np.std(per_class_f1[lbl]):.4f}")
    print(f"  F2:       {np.mean(per_class_f2[lbl]):.4f} ± {np.std(per_class_f2[lbl]):.4f}")

================ RESULTADO FINAL ================
Acurácia geral: 97.17 ± 1.59

=========== MÉTRICAS MACRO FINAIS ===========
Precisão: média = 0.9749 | desvio = 0.0145
Recall:   média = 0.9717 | desvio = 0.0159
F1:       média = 0.9715 | desvio = 0.0160
F2:       média = 0.9712 | desvio = 0.0161

=========== MÉTRICAS POR CLASSE (MÉDIA ± STD) ===========
Alergia
  Precisão: 1.0000 ± 0.0000
  Recall:   1.0000 ± 0.0000
  F1:       1.0000 ± 0.0000
  F2:       1.0000 ± 0.0000
Autismo
  Precisão: 0.9786 ± 0.0510
  Recall:   1.0000 ± 0.0000
  F1:       0.9885 ± 0.0275
  F2:       0.9952 ± 0.0115
Bronquite
  Precisão: 0.9202 ± 0.0723
  Recall:   0.9750 ± 0.0595
  F1:       0.9441 ± 0.0453
  F2:       0.9617 ± 0.0485
Caimbra
  Precisão: 0.9429 ± 0.0780
  Recall:   0.8917 ± 0.0795
  F1:       0.9143 ± 0.0647
  F2:       0.9001 ± 0.0712
Convulsao
  Precisão: 0.9929 ± 0.0311
  Recall:   1.0000 ± 0.0000
  F1:       0.9962 ± 0.0168
  F2:       0.9984 ± 0.0070
Diarreia
  Precisão: 0.9143 ± 0.0700
  

### Acurácia por conjunto

In [14]:
print("Conjunto | Acurácia geral")
for k, acc in enumerate(set_accs, start=1):
    print(f"{k:8d} | {acc * 100:.2f}")

Conjunto | Acurácia geral
       1 | 96.67
       2 | 100.00
       3 | 95.00
       4 | 96.67
       5 | 98.33
       6 | 98.33
       7 | 96.67
       8 | 95.00
       9 | 98.33
      10 | 96.67
      11 | 96.67
      12 | 96.67
      13 | 98.33
      14 | 100.00
      15 | 98.33
      16 | 98.33
      17 | 95.00
      18 | 95.00
      19 | 95.00
      20 | 98.33
